#### Faiss
Facebook AI Similarity Search (Faiss) is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load = TextLoader("speech.txt")

C:\Users\saran_s\AppData\Local\Temp\ipykernel_5924\90875960.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
d:\OneDrive - Maarga Systems Private Limited\Documents\Agentic-AI-BridgeCourse\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
docs = load.load()
print(docs)

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\nâ€¦\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness 

In [3]:
len(docs)

1

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
docs = text_splitter.split_documents(docs)
embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5", encode_kwargs={
        "normalize_embeddings": True
    })


C:\Users\saran_s\AppData\Local\Temp\ipykernel_5924\810291435.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5", encode_kwargs={
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3491.22it/s]


In [5]:
db=FAISS.from_documents(docs, embedding=embedding)
db

In [6]:
result = db.similarity_search("who is saran subramani?")
result

[Document(id='67bb4b1d-9563-4edf-b344-c09cc86fd479', metadata={'source': 'speech.txt'}, page_content='who is saran subramani? he is a good man'),
 Document(id='72eedd6d-d238-4f48-9f68-086c8088aee7', metadata={'source': 'speech.txt'}, page_content='one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.'),
 Document(id='a0a340f3-7068-4ca3-8934-d2f8b6bcf51c', metadata={'source': 'speech.txt'}, page_content='â€¦'),
 Document(id='c8e762de-280a-4d21-9e14-ec3feda81a81', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a')]

In [7]:
result[0].page_content

'who is saran subramani? he is a good man'

#### As a Retriever
We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [8]:
retriever = db.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000023EED8ABFE0>, search_kwargs={})

In [9]:
result = retriever.invoke("who is saran")
result[0].page_content

'who is saran subramani? he is a good man'

In [10]:
db.save_local("faiss_db")

In [12]:
new_db = db.load_local("faiss_db", embedding, allow_dangerous_deserialization=True)
result = new_db.similarity_search("saran's full name")
result

[Document(id='67bb4b1d-9563-4edf-b344-c09cc86fd479', metadata={'source': 'speech.txt'}, page_content='who is saran subramani? he is a good man'),
 Document(id='a0a340f3-7068-4ca3-8934-d2f8b6bcf51c', metadata={'source': 'speech.txt'}, page_content='â€¦'),
 Document(id='72eedd6d-d238-4f48-9f68-086c8088aee7', metadata={'source': 'speech.txt'}, page_content='one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.'),
 Document(id='c8e762de-280a-4d21-9e14-ec3feda81a81', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a')]

In [13]:
result[0].page_content

'who is saran subramani? he is a good man'